# 🧬 BraveNewAlgorithm.jl — BBOB Sphere Function Demo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cecimerelo/BraveNewAlgorithm.jl/blob/master/examples/BraveNewAlgorithm_BBOB_Sphere_Demo.ipynb)

> *"Community, Identity, Stability"* — the World State motto in Aldous Huxley's **Brave New World** (1932).

This notebook demonstrates **BraveNewAlgorithm.jl**, a Julia metaheuristic optimisation algorithm
whose design is directly inspired by the caste system from Huxley's dystopian novel.
We will optimise the **BBOB Sphere function** — the canonical benchmark for continuous black-box
optimisation — and compare the algorithm against a random-search baseline.

---

## 🚀 Google Colab — One-time Environment Setup

Google Colab runs **Python** by default. The cells below install Julia 1.x and all required
packages into the Colab VM. **Run the two setup cells once per session, then proceed.**

| Step | What to do |
|------|------------|
| 1 | Run **Cell 1** — installs Julia via `juliaup` (≈ 2 min) |
| 2 | Run **Cell 2** — installs Julia packages (≈ 3–5 min first time, faster on cache) |
| 3 | Run the remaining cells in order |

> **Tip:** Use *Runtime → Run all* after the two setup cells finish to execute the full demo automatically.

In [ ]:
%%bash
# ── Step 1: Install Julia via juliaup (the official Julia installer) ──────────
# This installs the latest stable Julia release.
# juliaup manages Julia versions much like rustup manages Rust.

if ! command -v julia &> /dev/null; then
    echo "Installing Julia..."
    curl -fsSL https://install.julialang.org | sh -s -- -y --default-channel release
    echo "Julia installed successfully."
else
    echo "Julia is already installed: $(julia --version)"
fi

# Make julia available in PATH for subsequent cells
export PATH="$HOME/.juliaup/bin:$PATH"
julia --version

In [ ]:
%%bash
# ── Step 2: Install the BraveNewAlgorithm.jl package and its dependencies ─────
# We install directly from GitHub. Julia's package manager (Pkg) handles
# all transitive dependencies automatically.

export PATH="$HOME/.juliaup/bin:$PATH"

julia -e '
using Pkg
println("Installing BraveNewAlgorithm.jl from GitHub...")
Pkg.add(url="https://github.com/cecimerelo/BraveNewAlgorithm.jl.git")

println("Installing Plots.jl for visualisation...")
Pkg.add("Plots")

println("Precompiling packages (this can take a few minutes on first run)...")
Pkg.precompile()
println("✅ All packages installed and precompiled.")
'

---

## 📖 What is BraveNewAlgorithm?

**BraveNewAlgorithm** is a population-based metaheuristic that keeps the
*exploration / exploitation balance* by assigning every individual in the
population to one of **five castes**, each with a distinct role in the
evolutionary process.

The design metaphor comes from Aldous Huxley's *Brave New World*:
in the fictional World State, embryos are chemically conditioned before
birth to belong to predetermined social strata — from the highly
intelligent **Alphas** down to the menial **Epsilons**.
The algorithm borrows this hierarchy to impose *different evolutionary
pressures* on different groups of solutions.

---

## 🏛️ The Caste System

After each generation the population is **sorted by fitness** and divided
into five groups:

| Caste | Role | Evolutionary operator | Typical % |
|-------|------|-----------------------|-----------|
| **ALPHA** | Elite solutions — *exploit* the best region | Crossover with another Alpha + low mutation | 10 % |
| **BETA** | High performers — support exploitation | Crossover with an Alpha + moderate mutation | 20 % |
| **GAMMA** | Mid-tier — steady local improvement | Hill-climbing (local search) | 40 % |
| **DELTA** | Below-average — diversify the search | Mutation only (medium rate) | 20 % |
| **EPSILON** | Low performers — pure exploration | Mutation only (high rate) | 10 % |

> **Design rule:** BETA must always be exactly twice the size of ALPHA
> (`BETA% = 2 × ALPHA%`).  This ensures that every Alpha individual always
> has a Beta partner for crossover.

---

## 🔄 Algorithm Flow

```
Initialise random population  (Fertilising Room)
        │
        ▼
Sort by fitness
        │
        ▼
┌─────────────────────────────────────────────────────────┐
│  HATCHERY — divide into castes                          │
│  ALPHA | BETA | GAMMA | DELTA | EPSILON                 │
└─────────────────────────────────────────────────────────┘
        │
        ▼
┌─────────────────────────────────────────────────────────┐
│  EVOLUTION — apply operators per caste                  │
│  • Alpha × Alpha  →  crossover + low mutation           │
│  • Beta  × Alpha  →  crossover + moderate mutation      │
│  • Gamma          →  hill-climbing (local search)        │
│  • Delta          →  mutation (medium)                  │
│  • Epsilon        →  mutation (high)                    │
└─────────────────────────────────────────────────────────┘
        │
        ▼
Elitism: best individual from previous generation
is always injected into the new population
        │
        ▼
Stopping criterion met?  ──Yes──▶  Return results
        │ No
        └──────────▶  next generation
```

The algorithm stops when the **best fitness has not improved for
`max_generations` consecutive generations**, which avoids wasting
evaluations when the search has stagnated.

---

## 🎯 The Optimisation Problem — BBOB Sphere Function

We use **BBOB function 1** (the *Sphere* function), the simplest benchmark
from the Black-Box Optimisation Benchmarking suite:

$$f(\mathbf{x}) = \sum_{i=1}^{d}\, (x_i - x_i^*)^2 + f^*$$

| Property | Value |
|----------|-------|
| Class | Separable, unimodal |
| Optimum $f^*$ | Problem-instance dependent (BBOB randomises it) |
| Search domain | $[-5,\, 5]^d$ |
| Dimensions | 5 (in this demo) |

Because the Sphere is unimodal and smooth, a good algorithm should
converge quickly and reproducibly — an ideal sanity check.

> We compare the algorithm against a **baseline** that simply picks
> the best individual from the *initial random population* with no
> evolution at all, to quantify how much the algorithm actually helps.

In [ ]:
%%bash
# ── Part 1: Baseline — best individual from a random population ────────────────
# We create an initial population but do NOT evolve it.
# This shows the quality of a purely random search.

export PATH="$HOME/.juliaup/bin:$PATH"

julia -e '
using BraveNewAlgorithm
using BlackBoxOptimizationBenchmarking

# ── Configuration ─────────────────────────────────────────────────────────────
# All five castes must be present.
# Constraint: ALPHA% == BETA% / 2
# Constraint: all percentages sum to 100
# Constraint: (ALPHA% * population_size / 100) must be even (pairs for crossover)

config = ConfigurationParametersEntity(
    5,    # chromosome_size      — 5-dimensional search space
    40,   # population_size      — 40 individuals in the population
    100,  # max_generations      — stop after 100 stagnant generations
    10,   # max_hillclimbing_steps — hill-climb budget for GAMMA caste
    Dict{String,Int}(
        "ALPHA"   => 10,   # 10 % → 4 individuals  (elite)
        "BETA"    => 20,   # 20 % → 8 individuals  (must be 2× ALPHA)
        "GAMMA"   => 40,   # 40 % → 16 individuals (hill-climbers)
        "DELTA"   => 20,   # 20 % → 8 individuals  (diversifiers)
        "EPSILON" => 10    # 10 % → 4 individuals  (explorers)
    ),
    Dict{String,Int}(
        "ALPHA"   => 3,    # very low mutation — protect elite solutions
        "BETA"    => 8,    # low mutation      — refine good solutions
        "GAMMA"   => 12,   # moderate          — local search complement
        "DELTA"   => 18,   # higher            — push diversification
        "EPSILON" => 25    # high              — pure exploration
    )
)

# ── Fitness function (BBOB Sphere, instance 1) ────────────────────────────────
bbob_fn          = BlackBoxOptimizationBenchmarking.BBOBFunctions[1]
fitness_baseline = FitnessFunction(bbob_fn)
search_range     = (-5.0, 5.0)
tolerance        = 1e-6
comparator       = (f_val, ff) -> f_val >= ff.f_opt + tolerance

baseline_model = PopulationModel(config, fitness_baseline, search_range, comparator)

# ── Build initial population (no evolution) ───────────────────────────────────
embryos       = multiple_fertilising_room(baseline_model)
best_baseline = best_element_of_population(embryos)

println("=" ^ 55)
println("  BASELINE: Best random individual (no evolution)")
println("=" ^ 55)
println("  Population size      : ", config.population_size)
println("  Dimensions           : ", config.chromosome_size)
println("  Best f-value found   : ", round(best_baseline.f_value, digits=6))
println("  Known optimum f*     : ", round(bbob_fn.f_opt, digits=6))
println("  Gap to optimum       : ", round(best_baseline.f_value - bbob_fn.f_opt, digits=6))
println("  Function evaluations : ", fitness_baseline.calls_counter)
'

The baseline shows what you get **for free** from random sampling alone.
With 40 individuals in a 5-dimensional space the random population already
finds a reasonable starting point, but there is typically a large gap
to the true optimum $f^*$.

Now let's see how much BraveNewAlgorithm improves on this.

In [ ]:
%%bash
# ── Part 2: Full BraveNewAlgorithm run ────────────────────────────────────────
# We use the same configuration as above but now let the algorithm evolve
# the population through multiple generations.

export PATH="$HOME/.juliaup/bin:$PATH"

julia -e '
using BraveNewAlgorithm
using BlackBoxOptimizationBenchmarking

config = ConfigurationParametersEntity(
    5,    # chromosome_size
    40,   # population_size
    100,  # max_generations (stagnation limit)
    10,   # max_hillclimbing_steps
    Dict{String,Int}(
        "ALPHA" => 10, "BETA" => 20, "GAMMA" => 40,
        "DELTA" => 20, "EPSILON" => 10
    ),
    Dict{String,Int}(
        "ALPHA" => 3, "BETA" => 8, "GAMMA" => 12,
        "DELTA" => 18, "EPSILON" => 25
    )
)

bbob_fn      = BlackBoxOptimizationBenchmarking.BBOBFunctions[1]
fitness_fn   = FitnessFunction(bbob_fn)
search_range = (-5.0, 5.0)
tolerance    = 1e-6
comparator   = (f_val, ff) -> f_val >= ff.f_opt + tolerance

model = PopulationModel(config, fitness_fn, search_range, comparator)

# ── Run ───────────────────────────────────────────────────────────────────────
println("Running BraveNewAlgorithm — please wait...")
t0 = time()
generation, results = brave_new_algorithm(model)
elapsed = round(time() - t0, digits=2)

best_fitness = minimum(results.F_Values)
f_opt        = bbob_fn.f_opt

println()
println("=" ^ 55)
println("  BNA RESULT")
println("=" ^ 55)
println("  Generations (stagnation): ", generation)
println("  Wall-clock time          : ", elapsed, " s")
println("  Function evaluations     : ", fitness_fn.calls_counter)
println("  Best f-value found       : ", round(best_fitness, digits=8))
println("  Known optimum f*         : ", round(f_opt, digits=8))
println("  Gap to optimum           : ", round(best_fitness - f_opt, digits=8))
println()

# ── Per-generation convergence table ─────────────────────────────────────────
println("Convergence (sampled every 10 % of recorded generations):")
println("-" ^ 45)
n     = length(results.F_Values)
step  = max(1, div(n, 10))
for i in 1:step:n
    gen   = results.Generations[i]
    f_val = results.F_Values[i]
    gap   = f_val - f_opt
    println("  Gen ", lpad(string(gen), 4), "  |  f = ", rpad(string(round(f_val, digits=6)), 12), "  |  gap = ", round(gap, sigdigits=2))
end
# Print the very last entry if not already shown
if n > 0 && (n - 1) % step != 0
    gen   = results.Generations[n]
    f_val = results.F_Values[n]
    gap   = f_val - f_opt
    println("  Gen ", lpad(string(gen), 4), "  |  f = ", rpad(string(round(f_val, digits=6)), 12), "  |  gap = ", round(gap, sigdigits=2))
end
' 2>/dev/null

### Understanding the Output

| Field | Meaning |
|-------|---------|
| **Generations (stagnation)** | The algorithm ran until the best fitness did not improve for this many consecutive generations |
| **Function evaluations** | Total number of calls to the Sphere function — a budget measure |
| **Best f-value** | The best fitness value achieved (lower = better for minimisation) |
| **Known optimum f*** | The true global minimum for this BBOB instance |
| **Gap to optimum** | How close we got; the stopping criterion fires at gap ≤ 10⁻⁶ |

You should see a dramatic reduction in the gap compared to the random baseline —
the caste-based evolution is working!

In [ ]:
%%bash
# ── Part 3: Side-by-side comparison of Baseline vs BNA ────────────────────────

export PATH="$HOME/.juliaup/bin:$PATH"

julia -e '
using BraveNewAlgorithm
using BlackBoxOptimizationBenchmarking

config = ConfigurationParametersEntity(
    5, 40, 100, 10,
    Dict{String,Int}("ALPHA" => 10, "BETA" => 20, "GAMMA" => 40,
                     "DELTA" => 20, "EPSILON" => 10),
    Dict{String,Int}("ALPHA" => 3,  "BETA" => 8,  "GAMMA" => 12,
                     "DELTA" => 18, "EPSILON" => 25)
)

bbob_fn  = BlackBoxOptimizationBenchmarking.BBOBFunctions[1]
range    = (-5.0, 5.0)
tol      = 1e-6
cmp      = (f, ff) -> f >= ff.f_opt + tol
f_opt    = bbob_fn.f_opt

# ── Baseline ─────────────────────────────────────────────────────────────────
ff_base   = FitnessFunction(bbob_fn)
embryos   = multiple_fertilising_room(PopulationModel(config, ff_base, range, cmp))
best_rand = best_element_of_population(embryos).f_value

# ── BNA ──────────────────────────────────────────────────────────────────────
ff_bna          = FitnessFunction(bbob_fn)
_, results      = brave_new_algorithm(PopulationModel(config, ff_bna, range, cmp))
best_bna        = minimum(results.F_Values)

gap_rand = best_rand - f_opt
gap_bna  = best_bna  - f_opt
pct_improvement = (gap_rand > 0) ? round((gap_rand - gap_bna) / gap_rand * 100, digits=1) : 0.0

println()
println("=" ^ 55)
println("  COMPARISON: Baseline vs BraveNewAlgorithm")
println("=" ^ 55)
println("  True optimum f*          : ", round(f_opt,     digits=6))
println()
println("  Random baseline:")
println("    best f-value           : ", round(best_rand, digits=6))
println("    gap to optimum         : ", round(gap_rand,  digits=6))
println("    evaluations            : ", ff_base.calls_counter)
println()
println("  BraveNewAlgorithm:")
println("    best f-value           : ", round(best_bna,  digits=6))
println("    gap to optimum         : ", round(gap_bna,   digits=6))
println("    evaluations            : ", ff_bna.calls_counter)
println()
println("  Gap improvement          : ", pct_improvement, " %")
println()
if gap_bna <= tol
    println("  ✅ BNA reached the optimum (gap ≤ tolerance).")
else
    println("  ℹ️  BNA did not reach exact optimum — gap is ",
            round(gap_bna, sigdigits=3), ".")
    println("     Try increasing max_generations or population_size.")
end
' 2>/dev/null

In [ ]:
%%bash
# ── Part 4: Convergence plot saved as PNG ─────────────────────────────────────
# We use Plots.jl (GR backend, headless) to generate a convergence curve.
# The PNG is saved to /tmp/bna_convergence.png and displayed in the next cell.

export PATH="$HOME/.juliaup/bin:$PATH"

julia -e '
using BraveNewAlgorithm
using BlackBoxOptimizationBenchmarking
using Plots
gr()   # GR backend works headlessly in Colab

config = ConfigurationParametersEntity(
    5, 40, 100, 10,
    Dict{String,Int}("ALPHA" => 10, "BETA" => 20, "GAMMA" => 40,
                     "DELTA" => 20, "EPSILON" => 10),
    Dict{String,Int}("ALPHA" => 3,  "BETA" => 8,  "GAMMA" => 12,
                     "DELTA" => 18, "EPSILON" => 25)
)

bbob_fn = BlackBoxOptimizationBenchmarking.BBOBFunctions[1]
ff      = FitnessFunction(bbob_fn)
range   = (-5.0, 5.0)
cmp     = (f, ff_) -> f >= ff_.f_opt + 1e-6

_, results = brave_new_algorithm(PopulationModel(config, ff, range, cmp))

f_opt = bbob_fn.f_opt
gaps  = max.(results.F_Values .- f_opt, 1e-12)   # ensure positive for log scale

p = plot(
    results.Generations, gaps,
    xlabel  = "Generation",
    ylabel  = "Gap to optimum (f - f*)",
    title   = "BraveNewAlgorithm — BBOB Sphere Convergence",
    lw      = 2,
    color   = :steelblue,
    legend  = false,
    yscale  = :log10,
    grid    = true,
    size    = (800, 450)
)
hline!([1e-6], linestyle = :dash, color = :red)

savefig(p, "/tmp/bna_convergence.png")
println("Plot saved to /tmp/bna_convergence.png")
' 2>/dev/null

In [ ]:
# Display the convergence plot inline in Colab
from IPython.display import Image
Image('/tmp/bna_convergence.png')

**Reading the convergence plot:**
The Y-axis uses a **log scale** so that the dramatic early improvements and
the fine-tuning near the optimum are both visible.
The red dashed line marks the **stopping tolerance** (gap ≤ 10⁻⁶);
once the best individual crosses that line the algorithm declares success.

---

## 🧪 Experiment — How Caste Percentages Affect Performance

One of the key design choices in BraveNewAlgorithm is **how to split the
population across castes**.  
The cell below runs the algorithm three times with different strategies and
compares the results:

| Strategy | Description | ALPHA / BETA / GAMMA / DELTA / EPSILON |
|----------|-------------|----------------------------------------|
| **Exploitation-heavy** | Invest most of the budget in the elite | 20 / 40 / 20 / 10 / 10 |
| **Balanced** | Equal weight across roles | 10 / 20 / 40 / 20 / 10 |
| **Exploration-heavy** | Favour diversity and global search | 10 / 20 / 20 / 25 / 25 |

In [ ]:
%%bash
export PATH="$HOME/.juliaup/bin:$PATH"

julia -e '
using BraveNewAlgorithm
using BlackBoxOptimizationBenchmarking

bbob_fn = BlackBoxOptimizationBenchmarking.BBOBFunctions[1]
range   = (-5.0, 5.0)
tol     = 1e-6
f_opt   = bbob_fn.f_opt

strategies = [
    # (label, chromosome_size, population_size, max_gen, max_hc,
    #  ALPHA, BETA, GAMMA, DELTA, EPSILON,
    #  alpha_mr, beta_mr, gamma_mr, delta_mr, eps_mr)
    ("Exploitation-heavy", 5, 100, 100, 10, 20, 40, 20, 10, 10, 3, 8, 12, 18, 25),
    ("Balanced",           5, 100, 100, 10, 10, 20, 40, 20, 10, 3, 8, 12, 18, 25),
    ("Exploration-heavy",  5, 100, 100, 10, 10, 20, 20, 25, 25, 3, 8, 12, 18, 25),
]

println("=" ^ 65)
println("  EXPERIMENT: Impact of Caste Distribution")
println("  (BBOB Sphere, 5D, population=100, stagnation limit=100 gen)")
println("=" ^ 65)
println()

for (label, chrom, pop, maxg, maxhc, a, b, g, d, e, amr, bmr, gmr, dmr, emr) in strategies
    config = ConfigurationParametersEntity(
        chrom, pop, maxg, maxhc,
        Dict{String,Int}("ALPHA" => a, "BETA" => b, "GAMMA" => g,
                         "DELTA" => d, "EPSILON" => e),
        Dict{String,Int}("ALPHA" => amr, "BETA" => bmr, "GAMMA" => gmr,
                         "DELTA" => dmr, "EPSILON" => emr)
    )
    ff     = FitnessFunction(bbob_fn)
    model  = PopulationModel(config, ff, range, (f_, ff_) -> f_ >= ff_.f_opt + tol)
    t0     = time()
    gen, results = brave_new_algorithm(model)
    elapsed = round(time() - t0, digits=2)
    best = minimum(results.F_Values)
    gap  = best - f_opt
    reached = gap <= tol ? "✅ optimal" : "gap=$(round(gap, sigdigits=2))"
    println("  Strategy : ", rpad(label, 20))
    println("    Castes (A/B/G/D/E) : $(a)/$(b)/$(g)/$(d)/$(e)")
    println("    Generations        : ", gen)
    println("    Evaluations        : ", ff.calls_counter)
    println("    Best gap to f*     : ", round(gap, sigdigits=3), "  — ", reached)
    println("    Time               : ", elapsed, " s")
    println()
end
' 2>/dev/null

### What to notice

* **Exploitation-heavy** configurations tend to converge faster on unimodal
  problems like the Sphere because they invest more budget in refining good
  solutions.  
* **Exploration-heavy** configurations are slower to converge here but would
  be better on multimodal landscapes (many local optima) where aggressive
  local search gets trapped.  
* **Balanced** is a safe default that works well across a wide range of
  problem types.

The right trade-off depends on your problem.  Use what you learn here as
intuition to tune for your own use-case.

---

## ⚙️ Customisation Guide

### Using a Different BBOB Function

```julia
# Rosenbrock (f2) — multimodal, classic benchmark
bbob_fn = BlackBoxOptimizationBenchmarking.BBOBFunctions[2]

# Ellipsoid (f3) — ill-conditioned
bbob_fn = BlackBoxOptimizationBenchmarking.BBOBFunctions[3]
```

### Tuning the Stopping Criterion

```julia
# Stop when very close to the optimum (tight)
comparator = (f_val, ff) -> f_val >= ff.f_opt + 1e-8

# Stop with a relaxed tolerance (faster)
comparator = (f_val, ff) -> f_val >= ff.f_opt + 1e-4
```

### Recommended Parameter Ranges

| Parameter | Recommended range | Notes |
|-----------|------------------|---------|
| `chromosome_size` | 2–50 | Problem dimensionality |
| `population_size` | 20–200 | Must allow even ALPHA & BETA groups |
| `max_generations` | 50–500 | Stagnation limit (not hard max) |
| `max_hillclimbing_steps` | 5–50 | GAMMA hill-climb budget per individual |
| ALPHA % | 5–25 | Always BETA% / 2 |
| BETA % | 10–50 | Always 2 × ALPHA% |

### Research Citation

If you use this algorithm in your research, please cite:

```bibtex
@Inbook{Merelo2022,
  author    = {Merelo, Cecilia and Merelo, Juan J. and Garc{\'{i}}a-Valdez, Mario},
  title     = {A Brave New Algorithm to Maintain the Exploration/Exploitation Balance},
  booktitle = {New Perspectives on Hybrid Intelligent System Design based on
               Fuzzy Logic, Neural Networks and Metaheuristics},
  year      = {2022},
  publisher = {Springer International Publishing},
  pages     = {305--316},
  doi       = {10.1007/978-3-031-08266-5_20}
}
```

---

*Notebook by GitHub Copilot, based on
[cecimerelo/BraveNewAlgorithm.jl](https://github.com/cecimerelo/BraveNewAlgorithm.jl).
Licence: GPL v3.*